# Milestone 2: Classical ML Baseline (XGBoost)

Feature extraction (MFCC + Chroma + Spectral) and XGBoost classification with noise augmentation.

In [ ]:
!pip install transformers wandb librosa xgboost -q

In [ ]:
import sys, os, random
sys.path.append('../src')
import numpy as np, pandas as pd, librosa
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score, accuracy_score, classification_report
import xgboost as xgb
import wandb

from utils import *
set_seed()
paths = get_paths()

## Feature Extraction (MFCC + Chroma + Spectral)

In [ ]:
noise_files_list = []
if paths['NOISE_DIR']:
    noise_files_list = [os.path.join(paths['NOISE_DIR'], f)
                        for f in os.listdir(paths['NOISE_DIR']) if f.endswith('.wav')]

print(f"Extracting features from training data (duration={DURATION}s)...")
X_train_ml = []
y_train_ml = []

for genre in tqdm(GENRES, desc="Genres"):
    genre_path = os.path.join(paths['STEMS_DIR'], genre)
    if not os.path.exists(genre_path):
        continue
    for song_folder in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song_folder)
        mixed = mix_stems_to_audio(song_path)

        # Noise augmentation with ESC-50
        if noise_files_list and random.random() < 0.5:
            noise_file = random.choice(noise_files_list)
            noise_y, _ = librosa.load(noise_file, sr=SAMPLE_RATE, duration=DURATION)
            num_frames = len(mixed)
            if len(noise_y) < num_frames:
                noise_y = np.pad(noise_y, (0, num_frames - len(noise_y)))
            else:
                noise_y = noise_y[:num_frames]
            signal_power = np.mean(mixed ** 2) + 1e-10
            noise_power = np.mean(noise_y ** 2) + 1e-10
            snr_db = random.uniform(5, 20)
            scale = np.sqrt(signal_power / (10 ** (snr_db / 10) * noise_power))
            mixed = mixed + scale * noise_y
            max_val = np.max(np.abs(mixed))
            if max_val > 0:
                mixed = mixed / max_val

        feat = extract_features_from_array(mixed, SAMPLE_RATE)
        X_train_ml.append(feat)
        y_train_ml.append(genre)

X_train_ml = np.array(X_train_ml)
print(f"\nFeature matrix shape: {X_train_ml.shape}")
print(f"Total samples: {len(y_train_ml)}")

## Train XGBoost + WandB

In [ ]:
wandb_login()
wandb.init(project="22f1001611-t12026", name="xgboost-mfcc-baseline")

le = LabelEncoder()
y_encoded = le.fit_transform(y_train_ml)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_ml, y_encoded, test_size=0.2, random_state=SEED, stratify=y_encoded
)

print("Training XGBoost...")
clf = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=SEED, use_label_encoder=False, eval_metric='mlogloss'
)
clf.fit(X_tr, y_tr)

y_pred = clf.predict(X_val)
val_acc = accuracy_score(y_val, y_pred)
val_f1 = f1_score(y_val, y_pred, average='macro')

print(f"\nXGBoost Results:")
print(f"  Validation Accuracy:  {val_acc:.4f}")
print(f"  Validation Macro F1:  {val_f1:.4f}")
print(f"\nPer-class report:")
print(classification_report(y_val, y_pred, target_names=GENRES))

wandb.log({"model": "XGBoost", "val_accuracy": val_acc, "val_macro_f1": val_f1,
           "n_features": X_train_ml.shape[1], "n_samples": len(y_train_ml)})
xgb_val_acc = val_acc
xgb_val_f1 = val_f1
wandb.finish()

## XGBoost Test Submission

In [ ]:
mashup_file_lookup = build_file_lookup(paths['MASHUPS_DIR'])

print("Extracting features from test mashups...")
test_df = pd.read_csv(paths['TEST_CSV'])
X_test_ml = []
test_ids_ml = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test features"):
    file_id = str(row['id'])
    file_path = find_test_file(file_id, mashup_file_lookup, row)

    if file_path is None:
        print(f"WARNING: File not found for id={file_id}, using zeros")
        X_test_ml.append(np.zeros(X_train_ml.shape[1]))
    else:
        feat = extract_features_from_file(file_path)
        X_test_ml.append(feat)
    test_ids_ml.append(int(file_id))

X_test_ml = np.array(X_test_ml)

y_test_pred = clf.predict(X_test_ml)
y_test_genres = le.inverse_transform(y_test_pred)

sub_xgb = pd.DataFrame({'id': test_ids_ml, 'genre': y_test_genres})
sub_xgb.to_csv('submission_xgb.csv', index=False)
print(f"\nXGBoost submission saved: submission_xgb.csv")
print(sub_xgb.head(10))
print(f"\nGenre distribution:")
print(sub_xgb['genre'].value_counts())